# 🌍 Notebook 3 — Comorian Translation AI
### Build a translation model: Comorian ↔ French ↔ English

---
## 📋 Prerequisites
- You need a **parallel corpus** (same sentences in Comorian + French and/or English)
- At least **5,000-10,000 parallel sentence pairs** for decent quality
- Run on **Kaggle** (30h/week free GPU) for training

## 📋 Steps
| Step | What happens |
|------|--------------|
| 1 | Install packages |
| 2 | Build parallel corpus |
| 3 | Load NLLB-200 base model |
| 4 | Prepare dataset |
| 5 | Fine-tune for translation |
| 6 | Test & save model |

> 💡 **Why NLLB-200?** Meta's No Language Left Behind model supports 200 languages
> including Swahili. Fine-tuning it for Comorian leverages its existing Bantu language knowledge.

## ✅ Step 1: Install Packages

In [ ]:
print('📦 Installing packages...')
!pip install transformers datasets accelerate sentencepiece sacrebleu -q
print('✅ All packages installed!')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🔧 Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

## ✅ Step 2: Build Your Parallel Corpus

A parallel corpus has the **same sentence** in multiple languages, side by side.

### How to build it (easiest methods):

| Method | Effort | Quality |
|--------|--------|--------|
| 🗣️ Translate yourself | High | ⭐⭐⭐ Best |
| 📖 Religious texts (Quran, Bible) | Low | ⭐⭐ Good |
| 📰 Government documents | Medium | ⭐⭐ Good |
| 🤖 Use GPT/Claude to help translate | Low | ⭐ Review needed |

### Format: `parallel_corpus.csv`
```csv
comorian,french,english
Salamualaikum warahmatullahi,Que la paix soit sur vous,Peace be upon you
Habar Zaho,Comment allez-vous,How are you
```

> 💡 Start with **100 sentences** and grow. Even 1,000 pairs can produce useful results.

In [ ]:
import pandas as pd
import os

CORPUS_PATH = '/content/parallel_corpus.csv'

print('📋 To get started:')
print()
print('   1. Create parallel_corpus.csv with columns: comorian, french, english')
print('   2. Upload it to this notebook')
print('   3. Re-run this cell')
print()

if os.path.exists(CORPUS_PATH):
    df = pd.read_csv(CORPUS_PATH)
    print(f'✅ Loaded parallel corpus: {len(df)} sentence pairs')
    print(f'   Columns: {list(df.columns)}')
    print(f'\n   Preview:')
    print(df.head(5).to_string(index=False))
else:
    print('⚠️  No parallel_corpus.csv found yet.')
    print('   Upload your file or create one with the format above.')

    # Create a starter template
    starter = pd.DataFrame({
        'comorian': [
            'Salamualaikum warahmatullahi wabarakatuh',
            'Habar Zaho',
            'Wami nitsaha ndriya',
            'Djeje lagno',
            'Mwana wangu',
        ],
        'french': [
            'Que la paix et la mis\u00e9ricorde de Dieu soient sur vous',
            'Comment allez-vous',
            'Je veux manger',
            'Comment tu t\'appelles',
            'Mon enfant',
        ],
        'english': [
            'Peace and mercy of God be upon you',
            'How are you',
            'I want to eat',
            'What is your name',
            'My child',
        ]
    })
    starter.to_csv(CORPUS_PATH, index=False)
    print(f'\n📝 Created starter template with {len(starter)} example pairs')
    print(f'   Edit {CORPUS_PATH} and add your translations!')
    df = starter

## ✅ Step 3: Load NLLB-200 Model

We use `facebook/nllb-200-distilled-600M` — small enough for free GPUs, powerful enough for translation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_ID = 'facebook/nllb-200-distilled-600M'

print(f'🔄 Loading {MODEL_ID}...')
print(f'   This may take 1-2 minutes on first run...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.to(device)

print(f'\n✅ NLLB-200 loaded!')
print(f'   Parameters: 600M')
print(f'   Languages: 200')
print(f'   Device: {device}')

## ✅ Step 4: Test Zero-Shot Translation (Before Fine-tuning)

Let's see how NLLB handles Comorian out of the box using Swahili as proxy.

In [ ]:
def translate(text, src_lang='swh_Latn', tgt_lang='fra_Latn', max_length=128):
    """Translate text using NLLB-200."""
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", max_length=max_length, truncation=True).to(device)
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            max_length=max_length,
        )
    return tokenizer.decode(generated[0], skip_special_tokens=True)

# Test with Comorian phrases (using Swahili code since Comorian isn't in NLLB)
test_phrases = [
    'Salamualaikum warahmatullahi wabarakatuh',
    'Habar Zaho',
    'Wami nitsaha ndriya',
]

print('🔬 Zero-shot translation test (Swahili\u2192French, before fine-tuning):')
print('=' * 60)
for phrase in test_phrases:
    fr = translate(phrase, 'swh_Latn', 'fra_Latn')
    en = translate(phrase, 'swh_Latn', 'eng_Latn')
    print(f'\n   🇰🇲 {phrase}')
    print(f'   🇫🇷 {fr}')
    print(f'   🇬🇧 {en}')

print('\n💡 These results will improve significantly after fine-tuning on Comorian data!')

## ✅ Step 5: Fine-tune on Your Parallel Corpus

> ⚠️ This step requires at least **1,000+ parallel pairs** for meaningful improvement.
> Start collecting translations now and come back when ready!

In [ ]:
from datasets import Dataset

MIN_PAIRS = 100

if len(df) < MIN_PAIRS:
    print(f'⚠️  Only {len(df)} pairs — need at least {MIN_PAIRS} to start fine-tuning.')
    print(f'   Keep building your parallel corpus!')
    print(f'\n📝 Ways to grow your corpus:')
    print(f'   • Translate common phrases yourself (most valuable)')
    print(f'   • Find bilingual Comorian/French texts online')
    print(f'   • Use religious texts with known translations')
    print(f'   • Ask native speakers to translate sentences')
else:
    print(f'✅ {len(df)} pairs — enough to start fine-tuning!')
    print(f'\n🔄 Preparing dataset...')

    # Prepare Comorian → French pairs
    train_data = []
    for _, row in df.iterrows():
        if pd.notna(row.get('comorian')) and pd.notna(row.get('french')):
            train_data.append({
                'source': str(row['comorian']),
                'target': str(row['french']),
            })

    dataset = Dataset.from_list(train_data)
    split = dataset.train_test_split(test_size=0.1, seed=42)
    print(f'   Train: {len(split["train"])} pairs')
    print(f'   Test:  {len(split["test"])} pairs')

    # Tokenize
    def preprocess(examples):
        tokenizer.src_lang = 'swh_Latn'
        model_inputs = tokenizer(
            examples['source'], max_length=128, truncation=True, padding='max_length'
        )
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                examples['target'], max_length=128, truncation=True, padding='max_length'
            )
        model_inputs['labels'] = labels['input_ids']
        return model_inputs

    tokenized = split.map(preprocess, batched=True)
    print('✅ Dataset tokenized and ready for training!')

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

if len(df) >= MIN_PAIRS:
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir='./nllb-comorian',
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,
        warmup_steps=100,
        max_steps=2000,
        fp16=True,
        evaluation_strategy='steps',
        eval_steps=200,
        save_steps=200,
        logging_steps=50,
        predict_with_generate=True,
        generation_max_length=128,
        load_best_model_at_end=True,
        push_to_hub=False,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized['train'],
        eval_dataset=tokenized['test'],
        data_collator=data_collator,
        tokenizer=tokenizer,
    )

    print('🚀 Starting fine-tuning on Comorian\u2192French...')
    trainer.train()
    print('\n🎉 Training complete!')
else:
    print('⚠️  Need more parallel pairs to train. Keep building your corpus!')

## ✅ Step 6: Test After Fine-tuning

In [ ]:
if len(df) >= MIN_PAIRS:
    print('🔬 Translation test AFTER fine-tuning:')
    print('=' * 60)
    for phrase in test_phrases:
        fr = translate(phrase, 'swh_Latn', 'fra_Latn')
        print(f'\n   🇰🇲 {phrase}')
        print(f'   🇫🇷 {fr}')

    # Save
    model.save_pretrained('./nllb-comorian-final')
    tokenizer.save_pretrained('./nllb-comorian-final')
    print(f'\n💾 Model saved to ./nllb-comorian-final/')
    print(f'\n📤 To share on HuggingFace:')
    print(f'   !huggingface-cli login')
    print(f'   trainer.push_to_hub()')
else:
    print('⚠️  Fine-tuning not done yet — build your parallel corpus first!')

---
## 🗺️ Growing Your Translation Corpus

### Quick wins for parallel data:
1. **Daily phrases** — Translate 10 common phrases per day
2. **Religious texts** — Quran/Bible translations exist in many languages
3. **Government** — Official Comorian documents often have French versions
4. **Social media** — Comorian speakers often code-switch with French
5. **Community** — Ask family/friends to translate sentences

### Targets:
| Pairs | Expected Quality |
|-------|------------------|
| 100-500 | Basic — common phrases only |
| 1,000-5,000 | Usable — simple conversations |
| 10,000-50,000 | Good — general translation |
| 100,000+ | Excellent — publication quality |

---
*Built for the Comorian (Shikomori) Language Preservation Project* 🇰🇲